# 01 · Synthetic Data Generation
No public dataset has multi-customer transactions **with a consistent running balance**, so the data is simulated.
Each customer gets a hidden *persona* that drives behaviour. The persona is never used by the analysis, only to **validate** it later.

The generator is `src/generate_data.py` and its settings are in `config/generator_config.yaml`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
import yaml
cfg = yaml.safe_load(open(ROOT / "config" / "generator_config.yaml"))
print(json.dumps({k: cfg[k] for k in ["seed", "n_customers", "start_date", "n_months", "persona_mix"]}, indent=1))

{
 "seed": 42,
 "n_customers": 120,
 "start_date": "2024-10-01",
 "n_months": 12,
 "persona_mix": {
  "stable_saver": 0.25,
  "comfortable_spender": 0.2,
  "emi_burdened": 0.15,
  "irregular_earner": 0.15,
  "lifestyle_creep": 0.1,
  "shock_event": 0.1,
  "student_early_career": 0.05
 }
}


## Persona design
| Persona | What the generator builds in |
|---|---|
| Stable Saver | fixed salary, savings 25–40%, SIPs, 3–8 months buffer |
| Comfortable Spender | fixed salary, thin 5–15% savings, high discretionary tilt |
| EMI-Burdened | loan EMIs at 40–55% of income (above typical lender FOIR limits) |
| Irregular Earner | freelance income, coefficient of variation 0.35–0.60 |
| Lifestyle Creep | spending grows 3–5% per month while income stays flat |
| Shock Event | stable until a medical bill, job loss or salary delay in Mar–Jul 2025 |
| Student / Early-career | family support + stipend, thin buffer |

**Realism rules:** salary on day 1–7; rent, EMI, SIP and insurance on fixed dates; weekend-weighted discretionary spend; Diwali, school-fee, summer-electricity and travel seasonality; payment method driven by amount and category.

**Borrowing:** when a debit would push the balance below ₹1,000, the customer borrows (credit card, instant-loan app or family) and repays app/card loans over the next three months. This keeps every balance consistent and creates a realistic debt-spiral signal.

In [3]:
raw = pd.read_csv(ROOT / "data/raw/transactions_raw.csv", dtype=str)
cust = pd.read_csv(ROOT / "data/raw/customers_raw.csv")
print(f"raw rows: {len(raw):,}   customers: {len(cust)}")
raw.head(8)

raw rows: 41,711   customers: 120


,transaction_id,customer_id,date,transaction_type,category,subcategory,merchant,amount,payment_method,income_expense,account_balance,recurring_flag,location,monthly_income,financial_goal
0,TXN0003582,C0015,2024-10-30 10:32:00,Debit,Groceries,Online Grocery,BigBasket,6293.32,UPI,Expense,91150.59,0,Online,96800.0,Wealth Creation
1,TXN0029986,C0064,2025-06-21 09:20:00,Debit,Subscriptions,OTT & Music,Spotify,119.0,UPI Autopay,Expense,404925.45,1,Online,62500.0,Debt Repayment
2,TXN0017766,C0064,2025-03-01 15:16:00,Debit,Utilities,Mobile Recharge,Vi,799.0,NaN,Expense,366983.32,1,Online,62500.0,Debt Repayment
3,TXN0003678,C0081,2024-10-31 15:46:00,Debit,Groceries,Online Grocery,Zepto,3311.38,Debit Card,Expense,524020.43,0,Online,100500.0,Wealth Creation
4,TXN0006861,C0094,2024-11-25 15:00:00,Debit,Groceries,Supermarket,Reliance Smart,6520.29,UPI,Expense,3331.28,0,Chennai,63800.0,Vehicle Purchase
5,TXN0015402,C0005,2025-02-07 19:34:00,Debit,NaN,Cafe,Cafe Coffee Day,5907.83,UPI,Expense,911086.98,0,Chandigarh,166200.0,Retirement
6,TXN0036984,C0070,2025-08-24 09:33:00,Debit,Food Delivery,Food Delivery App,Swiggy,3733.08,UPI,Expense,377511.24,0,Online,147900.0,Travel
7,TXN0028584,C0027,2025-06-07 14:20:00,Debit,Shopping,Electronics,Reliance Digital,8577.85,Net Banking,Expense,605180.18,0,Chennai,145300.0,Emergency Fund


In [4]:
cust.head()

,customer_id,age_band,city,city_tier,occupation_type,declared_monthly_income,financial_goal,has_active_loan
0,C0001,41-50,Coimbatore,Tier 2,Salaried,"96,800.00",Emergency Fund,True
1,C0002,31-40,Pune,Tier 1,Salaried,"126,700.00",Child Education,True
2,C0003,26-30,Hyderabad,Tier 1,Salaried,"93,500.00",Travel,True
3,C0004,21-25,Pune,Tier 1,Freelancer/Gig,"37,900.00",Vehicle Purchase,False
4,C0005,41-50,Chandigarh,Tier 2,Salaried,"166,200.00",Retirement,False


## Injected data-quality issues (logged so the cleaning step can be graded)

In [5]:
log = pd.read_csv(ROOT / "data/raw/injected_issues_log.csv")
log["issue_type"].value_counts().rename("rows").to_frame()

,rows
issue_type,
category_variant,1629
payment_method_variant,1234
mixed_date_format,1234
merchant_formatting,823
missing_category,411
missing_merchant,411
missing_payment_method,411
amount_as_text,411
exact_duplicate,411


In [6]:
# a few examples of what the dirty raw data looks like
ids = log.groupby("issue_type").head(1)["transaction_id"]
raw[raw["transaction_id"].isin(ids)][["transaction_id","date","category","merchant","amount","payment_method"]]

,transaction_id,date,category,merchant,amount,payment_method
3810,TXN0000431,2024-10-03 20:07:00,Utilities,ACT Fibernet,799.0,UPI Autopay
5523,TXN0033310,2052-07-20 11:43:00,Dining Out,Cafe Coffee Day,682.89,UPI
5608,TXN9000000,2025-02-04 10:15:00,Food Delivery,Swiggy,833.65,UPI
13720,TXN0022990,2025-04-16 20:00:00,Groceries,BigBasket,"₹1,503.08",Credit Card
15536,TXN0008944,12-12-2024 13:09:00,Utilities,Electricity Board,1433.52,UPI
30343,TXN0018694,2025-03-08 19:48:00,Food Delivery,Swiggy,695.2,BHIM UPI
34452,TXN0020995,2025-03-31 21:27:00,Transport,Indian Oil,-1292.5,NaN
35613,TXN0031750,2025-07-06 11:31:00,FOOD DELIVERY,Zomato,421.79,UPI
36871,TXN0024690,2025-05-04 10:47:00,Groceries,Zepto,303070.0,UPI
37629,TXN0015280,2025-02-07 06:03:00,NaN,SBI,16329.0,Auto-Debit (NACH)


## Data dictionary
See `data/data_dictionary.md` for every column, its type and its business meaning.